# Visualización de las señales

Antes de calcular cualquier cosa hay que ver los datos. En este notebook se cargan las grabaciones, se grafican los nueve canales del sensor y se comparan las actividades entre sí. El objetivo es aprender a leer las señales: qué valores tienen en reposo, cómo cambian con el movimiento y qué diferencias se ven a simple vista entre una actividad y otra. Esas diferencias son las que después se convierten en características para un modelo.

## Parámetros

In [ ]:
CARPETA = "datos/demo"   # datos/equipo_A o datos/equipo_B para los datos de cada equipo

## Carga de las grabaciones

`cargar_grabaciones` lee todos los CSV de la carpeta y toma la etiqueta del nombre de cada archivo. A cada actividad se le asigna un color que se mantiene en todas las gráficas del notebook.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from wt9011 import cargar_grabaciones

grabaciones = cargar_grabaciones(CARPETA)

ETIQUETAS = sorted(set(g["etiqueta"] for g in grabaciones))
COLORES = {e: f"C{i}" for i, e in enumerate(ETIQUETAS)}

# primera grabación de cada actividad, para las comparaciones
ejemplo = {}
for g in grabaciones:
    if g["etiqueta"] not in ejemplo:
        ejemplo[g["etiqueta"]] = g

filas = []
for g in grabaciones:
    filas.append([g["archivo"], g["etiqueta"], len(g["df"]), round(g["df"]["t"].iloc[-1], 1), round(g["fs"], 2)])

pd.DataFrame(filas, columns=["archivo", "etiqueta", "muestras", "duración [s]", "fs [Hz]"])

## Qué mide cada canal

Los ejes x, y, z están fijos al cuerpo del sensor, no al salón. Si el sensor gira, sus ejes giran con él.

El acelerómetro (ax, ay, az) mide la aceleración del movimiento más la gravedad. Con el sensor en reposo, el eje que apunta hacia arriba marca 1 g y los otros dos, 0. Si se inclina, la gravedad se reparte entre los ejes; por eso el nivel promedio de cada eje indica la orientación del sensor.

El giroscopio (gx, gy, gz) mide la velocidad de rotación alrededor de cada eje. En reposo marca 0 en los tres, sin importar la orientación.

Los ángulos los calcula el propio sensor combinando acelerómetro, giroscopio y magnetómetro. Roll es la rotación alrededor de x y pitch alrededor de y; los dos usan la gravedad como referencia y son confiables. Yaw, la rotación alrededor del eje vertical, depende del magnetómetro, que se altera cerca de metales y aparatos electrónicos; en interiores conviene tratarlo con cuidado.

## Una grabación completa

`INDICE` selecciona la grabación según el orden de la tabla anterior.

In [ ]:
INDICE = 0

g = grabaciones[INDICE]
df = g["df"]

grupos = [(["ax", "ay", "az"], "Aceleración [g]"),
          (["gx", "gy", "gz"], "Velocidad angular [°/s]"),
          (["roll", "pitch", "yaw"], "Ángulo [°]")]

fig, ejes = plt.subplots(3, 1, figsize=(12, 7), sharex=True)
for ax, (canales, unidad) in zip(ejes, grupos):
    for c in canales:
        ax.plot(df["t"], df[c], lw=0.8, label=c)
    ax.set_ylabel(unidad)
    ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1))
    ax.grid(alpha=0.3)
ejes[0].set_title(f"{g['archivo']}  ({g['etiqueta']})")
ejes[-1].set_xlabel("Tiempo [s]")
plt.tight_layout()
plt.show()

Al leer una grabación conviene fijarse en tres cosas. El nivel alrededor del cual se mueve cada eje de aceleración, que indica cómo estaba orientado el sensor. La amplitud de las oscilaciones, que indica qué tan intenso es el movimiento. Y si hay un patrón que se repite: en ese caso la actividad es periódica y el tiempo entre repeticiones da su frecuencia, por ejemplo la de los pasos al caminar.

## Magnitud de la aceleración y de la velocidad angular

Los valores de cada eje dependen de cómo se colocó el sensor. La magnitud del vector no depende de la orientación:

$$|a| = \sqrt{a_x^2 + a_y^2 + a_z^2} \qquad |\omega| = \sqrt{g_x^2 + g_y^2 + g_z^2}$$

En reposo |a| vale 1 g y |ω| vale 0. Con movimiento, |a| oscila alrededor de 1 g. Para distinguir actividades las magnitudes son más robustas que los ejes por separado, porque si el sensor queda un poco girado en la muñeca la magnitud no cambia. Se agregan como dos columnas nuevas a todas las grabaciones.

In [ ]:
for g in grabaciones:
    d = g["df"]
    d["a_mag"] = np.sqrt(d["ax"]**2 + d["ay"]**2 + d["az"]**2)
    d["g_mag"] = np.sqrt(d["gx"]**2 + d["gy"]**2 + d["gz"]**2)

g = grabaciones[INDICE]
df = g["df"]
color = COLORES[g["etiqueta"]]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 4.5), sharex=True)
ax1.plot(df["t"], df["a_mag"], lw=0.8, color=color)
ax1.axhline(1, color="k", lw=0.8, ls="--")
ax1.set(ylabel="|a| [g]", title=f"{g['archivo']}  ({g['etiqueta']})")
ax2.plot(df["t"], df["g_mag"], lw=0.8, color=color)
ax2.set(ylabel="|ω| [°/s]", xlabel="Tiempo [s]")
for ax in (ax1, ax2):
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Comparación entre actividades

Una grabación de cada actividad, los primeros `VENTANA` segundos, con la misma escala vertical en todas las filas para que la comparación sea justa.

In [ ]:
VENTANA = 5   # segundos

fig, ejes = plt.subplots(len(ETIQUETAS), 2, figsize=(12, 2.2 * len(ETIQUETAS)), sharex=True, sharey="col")
for i, etiqueta in enumerate(ETIQUETAS):
    d = ejemplo[etiqueta]["df"]
    d = d[d["t"] <= VENTANA]
    ejes[i, 0].plot(d["t"], d["a_mag"], lw=0.8, color=COLORES[etiqueta])
    ejes[i, 1].plot(d["t"], d["g_mag"], lw=0.8, color=COLORES[etiqueta])
    ejes[i, 0].set_ylabel(etiqueta)

for ax in ejes.flat:
    ax.grid(alpha=0.3)
ejes[0, 0].set_title("|a| [g]")
ejes[0, 1].set_title("|ω| [°/s]")
ejes[-1, 0].set_xlabel("Tiempo [s]")
ejes[-1, 1].set_xlabel("Tiempo [s]")
plt.tight_layout()
plt.show()

Las diferencias entre actividades se pueden describir con las mismas tres ideas de antes: el nivel medio de la señal, qué tanto varía y si tiene un ritmo. Cada una tiene una medida numérica que se calculará más adelante: la media, la desviación estándar y la frecuencia dominante del espectro.

## Distribución de valores por actividad

Aquí se juntan todas las grabaciones de cada actividad. Cada caja abarca la mitad central de los valores y la línea dentro de ella es la mediana. Si las cajas de dos actividades casi no se traslapan, una sola medida ya permite distinguirlas; si se traslapan mucho, hará falta combinar varias.

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(12, 3.5))

for ax, columna, unidad in [(ejes[0], "a_mag", "|a| [g]"), (ejes[1], "g_mag", "|ω| [°/s]")]:
    valores = []
    for etiqueta in ETIQUETAS:
        señales = [g["df"][columna] for g in grabaciones if g["etiqueta"] == etiqueta]
        valores.append(np.concatenate(señales))

    cajas = ax.boxplot(valores, tick_labels=ETIQUETAS, patch_artist=True, showfliers=False,
                       medianprops={"color": "k"})
    for caja, etiqueta in zip(cajas["boxes"], ETIQUETAS):
        caja.set(facecolor=COLORES[etiqueta], alpha=0.6)
    ax.set_ylabel(unidad)
    ax.grid(alpha=0.3, axis="y")

plt.tight_layout()
plt.show()

## Revisión de calidad

Antes de procesar las grabaciones se revisa que no tengan problemas de adquisición:

- Huecos: si entre dos paquetes pasan más de 5 periodos de muestreo, se perdieron datos en la transmisión.
- Saturación: si algún eje llega al límite del rango (±16 g o ±2000 °/s), el valor real era mayor y la señal quedó recortada.
- Frecuencia de muestreo: debe ser parecida en todas las grabaciones.

Una grabación con huecos o saturación se repite o se descarta.

In [ ]:
filas = []
for g in grabaciones:
    d = g["df"]
    huecos = int(np.sum(np.diff(d["t_llegada"]) > 5 / g["fs"]))
    saturado = (d[["ax", "ay", "az"]].abs().max().max() > 15.9 or
                d[["gx", "gy", "gz"]].abs().max().max() > 1990)
    filas.append({"archivo": g["archivo"],
                  "fs [Hz]": round(g["fs"], 2),
                  "huecos": huecos,
                  "|a| máx [g]": round(d["a_mag"].max(), 2),
                  "|ω| máx [°/s]": round(d["g_mag"].max(), 1),
                  "saturación": "sí" if saturado else "no"})

pd.DataFrame(filas)

En `04_preprocesamiento` se limpian las señales: se separa la gravedad del movimiento y se filtra el ruido.